[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/calculus/c08_tank.ipynb)

# A tank and its tap: why integration undoes differentiation

Archimedes needed a whole book of geometry, twenty-four propositions, to find one curved area. Today a student finds
the same area in two lines. What changed is one loop: the water in a tank is the area under the tap's flow, and the
speed the water rises is the flow right now.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button at the top) to run it. Run each cell in turn, from the top down. When sliders appear, drag them and watch.
Nothing can break: if something looks odd, run everything again from the top. It works in your browser (no install)
and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look: one style for every figure in this notebook. Run this cell first.
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

SLATE, SLATE2 = "#1d2925", "#25332e"        # the slate ground
CHALK, DIM, FAINT = "#ece9e0", "#7e8e88", "#33433e"
CORAL = "#ef8a68"                           # the one key thing in each figure
TEAL, MINT, LILAC = "#8fcfc4", "#a9dca6", "#c3a9e6"   # the flow · slopes and rates · water and areas

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

MINUTES = np.linspace(0, 6, 601)            # one run of the tap: six minutes, in hundredths of a minute

def smooth_through(xs, ys, t):
    """A smooth curve through every point you set: a cubic spline (pieces of cubics joined without a kink). It draws
    any quadratic or cubic exactly, so a level of t² gives a flow of exactly 2t."""
    x, y = np.asarray(xs, float), np.asarray(ys, float)
    n, h = len(x) - 1, np.diff(np.asarray(xs, float))
    A, r = np.zeros((n + 1, n + 1)), np.zeros(n + 1)       # unknowns: the curve's bend (second slope) at each point
    for i in range(1, n):
        A[i, i - 1:i + 2] = h[i - 1], 2 * (h[i - 1] + h[i]), h[i]
        r[i] = 6 * ((y[i + 1] - y[i]) / h[i] - (y[i] - y[i - 1]) / h[i - 1])
    A[0, :3] = h[1], -(h[0] + h[1]), h[0]                  # "not-a-knot" ends: the first two pieces are one cubic,
    A[n, n - 2:] = h[n - 1], -(h[n - 2] + h[n - 1]), h[n - 2]   # and so are the last two
    M = np.linalg.solve(A, r)
    k = np.clip(np.searchsorted(x, t) - 1, 0, n - 1)
    a, b = x[k + 1] - t, t - x[k]
    return (M[k] * a**3 + M[k + 1] * b**3) / (6 * h[k]) + (y[k] / h[k] - M[k] * h[k] / 6) * a + (y[k + 1] / h[k] - M[k + 1] * h[k] / 6) * b

def pile_up(t, flow):
    """The water in the tank: add flow × a tiny bit of time, strip by strip (the area under the flow, so far)."""
    return np.concatenate([[0.0], np.cumsum((flow[1:] + flow[:-1]) / 2 * np.diff(t))])

def slope_of(t, level):
    """How fast the level rises: its slope, from neighbouring points."""
    return np.gradient(level, t)

def draw_tank(ax, litres, full=60, flowing=0.0):
    """A glass tank with a tap above it, holding `litres` (out of `full`)."""
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.4); ax.axis("off"); ax.grid(False)
    ax.add_patch(Rectangle((0.15, 0.05), 0.7, 0.85, fill=False, ec=CHALK, lw=2.5))
    lv = 0.85 * max(0.0, min(1.0, litres / full))
    ax.add_patch(Rectangle((0.16, 0.06), 0.68, max(0.0, lv - 0.01), color=LILAC, alpha=0.45, lw=0))
    ax.plot([0.16, 0.84], [0.05 + lv] * 2, color=CORAL, lw=3)
    ax.plot([0.0, 0.34, 0.34], [1.2, 1.2, 1.1], color=CHALK, lw=5, solid_capstyle="butt")   # the tap's pipe and spout
    ax.plot([0.29, 0.39], [1.24, 1.24], color=CHALK, lw=3)                                  # its handle
    if flowing > 0.02: ax.plot([0.34, 0.34], [1.09, 0.07 + lv], color=TEAL, lw=1 + 3 * min(1, flowing), alpha=0.85)
    ax.text(0.62, 0.05 + lv + 0.04, f"{litres:.1f} L", ha="center", color=CHALK, fontsize=15)

## 1 · Shape a flow, and watch the water grow

The tap starts off and ends off. In between, you choose how hard it pours: each of the first five sliders sets the
flow at one minute, in **litres a minute**, and a smooth flow runs through your choices (never below zero: a tap
can't pour less than nothing). The last slider is the clock. Drag it, and watch the tank fill.

The water in the tank is the **area under the flow** (lilac): every moment adds flow × a tiny bit of time.

In [ ]:
def tank_from_flow(minute_1=5.0, minute_2=8.0, minute_3=9.0, minute_4=8.0, minute_5=5.0, now=1.0):
    flow = np.clip(smooth_through(range(7), [0, minute_1, minute_2, minute_3, minute_4, minute_5, 0], MINUTES), 0, None)
    water = pile_up(MINUTES, flow)
    k = int(round(now * 100))
    fig = plt.figure(figsize=(11, 4.7))
    g = fig.add_gridspec(2, 2, width_ratios=[1, 2.6], hspace=0.45)
    tank, top, bot = fig.add_subplot(g[:, 0]), fig.add_subplot(g[0, 1]), fig.add_subplot(g[1, 1])
    draw_tank(tank, water[k], full=max(60, 1.1 * water.max()), flowing=flow[k] / 12)
    tank.set_title(f"after {now:.1f} min")
    top.plot(MINUTES, flow, color=TEAL)
    top.fill_between(MINUTES[:k + 1], flow[:k + 1], color=LILAC, alpha=0.45)
    top.plot([1, 2, 3, 4, 5], [minute_1, minute_2, minute_3, minute_4, minute_5], "o", color=TEAL)
    top.set_ylabel("L / min"); top.set_xlim(0, 6); top.set_ylim(min(0, flow.min()) - 0.5, max(12, flow.max()) + 0.5)
    top.set_title(f"the flow · shaded area = {water[k]:.1f} litres")
    bot.plot(MINUTES, water, color=LILAC, alpha=0.35)
    bot.plot(MINUTES[:k + 1], water[:k + 1], color=LILAC)
    bot.plot([now], [water[k]], "o", color=CORAL, ms=10)
    bot.set_xlabel("minutes"); bot.set_ylabel("litres"); bot.set_xlim(0, 6)
    bot.set_title("the water in the tank")
    plt.show()

interact(tank_from_flow,
         minute_1=FloatSlider(value=5, min=0, max=12, step=1, description="flow at 1"),
         minute_2=FloatSlider(value=8, min=0, max=12, step=1, description="flow at 2"),
         minute_3=FloatSlider(value=9, min=0, max=12, step=1, description="flow at 3"),
         minute_4=FloatSlider(value=8, min=0, max=12, step=1, description="flow at 4"),
         minute_5=FloatSlider(value=5, min=0, max=12, step=1, description="flow at 5"),
         now=FloatSlider(value=1, min=0, max=6, step=0.5, description="minutes"));

**Notice.** At six minutes the tap is off again, but the tank is at its fullest. Zero flow doesn't mean zero water.
It means the water has stopped rising. Pour harder in one minute, and the water graph climbs more steeply there.

## 2 · Look the other way: how fast is the water rising?

Here the flow is **6t − t²** litres a minute: it peaks at 9 at three minutes and is back to 0 at six. The mint
line touches the water graph at the time you choose. Its slope is how fast the water is rising right then.

In [ ]:
def rising(at=3.0):
    t = MINUTES
    flow, water = 6 * t - t**2, 3 * t**2 - t**3 / 3
    w, f = 3 * at**2 - at**3 / 3, 6 * at - at**2
    rate = slope_of(t, water)[int(round(at * 100))]
    fig, (ax, bx) = plt.subplots(1, 2, figsize=(11.5, 4.4))
    ax.plot(t, water, color=LILAC); ax.plot([at], [w], "o", color=CORAL, ms=9)
    span = np.array([-1.2, 1.2])
    ax.plot(at + span, w + rate * span, color=MINT, lw=2.5)
    ax.set_xlabel("minutes"); ax.set_ylabel("litres"); ax.set_title(f"water: rising at {rate:.2f} L/min")
    bx.plot(t, flow, color=TEAL); bx.plot([at], [f], "o", color=CORAL, ms=9)
    bx.set_xlabel("minutes"); bx.set_ylabel("L / min"); bx.set_title(f"flow at {at:.2f} min: {f:.2f} L/min")
    plt.tight_layout(); plt.show()

interact(rising, at=FloatSlider(value=3, min=0, max=6, step=0.25, description="minutes"));

**Notice.** The two numbers in the titles always agree. The rate the water rises, right now, is the flow right now.
At six minutes both are zero: the water graph is flat at its top.

## 3 · The sliver: on any curve at all

Shade the area under a curve from 0 up to x. Then push x a small step to the right. What gets added is a thin sliver
(coral), about as tall as the curve at x. So the area grows at the rate of the height you're adding.

In [ ]:
def sliver(x=2.0, step=0.3):
    f = lambda s: s**2
    area = lambda s: s**3 / 3
    s = np.linspace(0, 3, 400)
    fig, ax = plt.subplots(figsize=(9, 4.6))
    ax.plot(s, f(s), color=TEAL, label="y = x²")
    m = s <= x
    ax.fill_between(s[m], f(s[m]), color=LILAC, alpha=0.4, label=f"area from 0 to x = {area(x):.3f}")
    m2 = (s >= x) & (s <= x + step)
    ax.fill_between(s[m2], f(s[m2]), color=CORAL, alpha=0.75, label="the new sliver")
    grow = (area(x + step) - area(x)) / step
    ax.set_title(f"the area grew by {area(x + step) - area(x):.3f}, about {grow:.2f} × the step · height at x: {f(x):.2f}")
    ax.set_xlim(0, 3.2); ax.set_ylim(0, 10); ax.legend(loc="upper left")
    plt.show()

interact(sliver, x=FloatSlider(value=2, min=0.5, max=2.5, step=0.1, description="x"),
         step=FloatSlider(value=0.3, min=0.01, max=0.5, step=0.01, description="step"));

**Notice.** Make the step tiny. The growth rate closes in on the height: at x = 2, both go to 4.

## 4 · Backwards: shape a water level, and see the flow it needs

Now you choose the water instead. Each slider sets how many litres are in the tank at one minute (it starts empty).
The flow it needs is the **slope** of the water graph. Where the water falls, the flow is negative: the tap has
become a drain.

In [ ]:
def flow_from_level(minute_1=1.0, minute_2=4.0, minute_3=9.0, minute_4=16.0, minute_5=25.0, minute_6=36.0):
    level = smooth_through(range(7), [0, minute_1, minute_2, minute_3, minute_4, minute_5, minute_6], MINUTES)
    flow = slope_of(MINUTES, level)
    fig, (ax, bx) = plt.subplots(2, 1, figsize=(10, 4.9), sharex=True)
    ax.plot(MINUTES, level, color=LILAC)
    ax.plot(range(1, 7), [minute_1, minute_2, minute_3, minute_4, minute_5, minute_6], "o", color=LILAC)
    ax.set_ylabel("litres"); ax.set_title("the water level you shaped")
    bx.plot(MINUTES, flow, color=TEAL)
    bx.axhline(0, color=DIM, lw=1)
    bx.fill_between(MINUTES, flow, where=flow < 0, color=CORAL, alpha=0.5, label="water out (a drain)")
    bx.set_xlabel("minutes"); bx.set_ylabel("L / min"); bx.set_xlim(0, 6)
    bx.set_title(f"the flow it needs · at the end: {flow[-1]:.1f} L/min")
    if (flow < 0).any(): bx.legend(loc="upper left")
    plt.tight_layout(); plt.show()

interact(flow_from_level, **{f"minute_{k}": FloatSlider(value=k * k, min=0, max=40, step=1, description=f"water at {k}")
                             for k in range(1, 7)});

**Notice.** With the starting values the water is t², and the flow it needs is the straight line 2t. Slice a flow and
add it up: you get a level. Take the slope of a level: you get the flow back. One loop.

## 5 · Two readings of big F

The second half of the loop: **the integral from a to b of f equals F(b) − F(a)**, for any F whose slope is f. Below,
the strips are added one by one (the slow way) and F is read twice (the fast way). Add a constant to F: both
readings go up by it, and it cancels.

In [ ]:
CASES = {"2t  (the first tank)": (lambda x: 2 * x, lambda x: x**2, (0, 4)),
         "1 − x²  (Archimedes' shape)": (lambda x: 1 - x**2, lambda x: x - x**3 / 3, (-1, 1)),
         "sin x  (one arch)": (np.sin, lambda x: -np.cos(x), (0, math.pi))}

def two_readings(curve="1 − x²  (Archimedes' shape)", strips=40, plus=0.0):
    f, F, (a, b) = CASES[curve]
    x = np.linspace(a, b, 400)
    edges = np.linspace(a, b, strips + 1)
    mids = (edges[:-1] + edges[1:]) / 2
    by_strips = float(np.sum(f(mids) * np.diff(edges)))
    by_F = (F(b) + plus) - (F(a) + plus)
    fig, ax = plt.subplots(figsize=(9, 4.4))
    ax.bar(mids, f(mids), width=np.diff(edges), color=LILAC, alpha=0.45, edgecolor=SLATE, lw=0.5)
    ax.plot(x, f(x), color=TEAL)
    ax.axhline(0, color=DIM, lw=1)
    ax.set_title(f"{strips} strips: {by_strips:.5f}    ·    F(b) − F(a) = {F(b) + plus:.4f} − ({F(a) + plus:.4f}) = {by_F:.5f}".replace("-", "−"))
    plt.show()

interact(two_readings, curve=Dropdown(options=list(CASES), value="1 − x²  (Archimedes' shape)", description="curve"),
         strips=IntSlider(value=40, min=4, max=400, step=4, description="strips"),
         plus=FloatSlider(value=0, min=-10, max=10, step=1, description="+ C"));

**Notice.** More strips bring the slow total closer and closer to the two-reading answer. The "+ C" slider changes
both readings, never their difference.

## 6 · Slice it from the other end

The integral from 0 to π/2 of sin x ÷ (sin x + cos x) looks impossible. But counting the same strips from the other
end turns every sine into a cosine. Both versions have the same area, and added together the curve is just 1.

In [ ]:
x = np.linspace(0, math.pi / 2, 400)
g = np.sin(x) / (np.sin(x) + np.cos(x))
h = np.cos(x) / (np.sin(x) + np.cos(x))
mid = lambda f, n=2000: float(np.mean(f((np.arange(n) + 0.5) * (math.pi / 2) / n)) * math.pi / 2)
I1 = mid(lambda s: np.sin(s) / (np.sin(s) + np.cos(s)))
I2 = mid(lambda s: np.cos(s) / (np.sin(s) + np.cos(s)))
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.fill_between(x, g, color=LILAC, alpha=0.35, label=f"sin ÷ (sin + cos): area {I1:.6f}")
ax.plot(x, g, color=TEAL); ax.plot(x, h, color=TEAL, ls="--", label=f"cos ÷ (sin + cos): area {I2:.6f}")
ax.plot(x, g + h, color=CORAL, label="the two added: 1")
ax.set_xticks([0, math.pi / 4, math.pi / 2]); ax.set_xticklabels(["0", "π/4", "π/2"])
ax.set_ylim(0, 1.2); ax.legend(loc="lower right"); plt.show()
print(f"each area = {I1:.6f};  π/4 = {math.pi / 4:.6f}")

**Notice.** The dashed curve is the solid one flipped left to right: the same strips, counted backwards.

## 7 · Check your answer

The challenge from the video: **the integral from 0 to π/2 of sin²x dx**. Work it out on paper first (hint: slice it
from the other end, then add). Then run this cell. It asks sympy for an antiderivative, checks its slope is the
curve you started with, and reads it at both ends.

In [ ]:
import sympy as sp
X = sp.symbols("x")
f = sp.sin(X) ** 2
F = sp.integrate(f, X)
print("an antiderivative F:", F)
print("its slope, simplified:", sp.simplify(sp.diff(F, X)))
print("F(π/2) − F(0) =", sp.simplify(F.subs(X, sp.pi / 2) - F.subs(X, 0)))
your_answer = sp.pi / 4                  # ← type your answer here, e.g. sp.pi / 4 or 0.785
print("your answer is", "right" if abs(float(your_answer) - float(sp.pi / 4)) < 1e-3 else "not yet: try again")

## Sources
T. L. Heath, *The Works of Archimedes* (Cambridge, 1897), *Quadrature of the Parabola*, Props. 1–24. MacTutor
History of Mathematics, "Isaac Barrow". M. Nauenberg, "Barrow and Leibniz on the fundamental theorem of the
calculus" (arXiv:1111.6145). NCERT class 12, Integrals (properties of definite integrals). All retrieved 2026-10-24.
Every number in this notebook is computed as you run it.